# Postgres Repo Glob/Grep Demo

This notebook connects to the Postgres-backed VFS snapshot created by `scripts/load_repo_into_postgres_vfs.py` and lets you run `glob` and `grep` queries against the mounted repo.

Default database:

- `postgresql+asyncpg://localhost/grover_repo_vfs`
- mount path: `/repo`


In [1]:
import os

from sqlalchemy.ext.asyncio import create_async_engine

from vfs import VFSClient
from vfs.backends.postgres import PostgresFileSystem

DB_URL = os.environ.get("GROVER_REPO_DB_URL", "postgresql+asyncpg://localhost/grover_repo_vfs")
MOUNT = "/repo"

engine = create_async_engine(DB_URL, echo=False)
client = VFSClient()
client.add_mount(MOUNT, PostgresFileSystem(engine=engine))

print(DB_URL)


postgresql+asyncpg://localhost/grover_repo_vfs


In [2]:
def _mounted_pattern(pattern: str) -> str:
    return pattern if pattern.startswith(MOUNT) else f"{MOUNT}/{pattern.lstrip('/')}"


def preview(result, limit: int = 20):
    trimmed = result.model_copy(update={"candidates": result.candidates[:limit]})
    print(trimmed.to_str())
    if len(result) > limit:
        print(f"\n... showing {limit} of {len(result)} candidates")
    return result


def repo_glob(pattern: str, **kwargs):
    return client.glob(_mounted_pattern(pattern), **kwargs)


def repo_grep(pattern: str, paths=(MOUNT,), **kwargs):
    return client.grep(pattern, paths=paths, **kwargs)


In [3]:
# Example glob queries
preview(repo_glob("**/*.py"), limit=25)


/repo/examples/cli_query_demo.py
/repo/examples/demo_dbfs.py
/repo/examples/main.py
/repo/scripts/bench_bm25.py
/repo/scripts/bench_bm25_comparison.py
/repo/scripts/bm25_comparison.py
/repo/scripts/bump_version.py
/repo/scripts/demo_content_gram_index.py
/repo/scripts/postgres_repo_cli_probe.py
/repo/scripts/store_repo.py
/repo/src/vfs/__init__.py
/repo/src/vfs/backends/__init__.py
/repo/src/vfs/backends/database.py
/repo/src/vfs/backends/mssql.py
/repo/src/vfs/backends/postgres.py
/repo/src/vfs/base.py
/repo/src/vfs/bm25.py
/repo/src/vfs/client.py
/repo/src/vfs/columns.py
/repo/src/vfs/databricks_store.py
/repo/src/vfs/embedding.py
/repo/src/vfs/exceptions.py
/repo/src/vfs/graph/__init__.py
/repo/src/vfs/graph/protocol.py
/repo/src/vfs/graph/rustworkx.py

... showing 25 of 83 candidates


VFSResult(success=True, errors=[], function='glob', candidates=[Candidate(path='/repo/examples/cli_query_demo.py', kind='file', lines=None, content=None, size_bytes=3543, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 347115, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/examples/demo_dbfs.py', kind='file', lines=None, content=None, size_bytes=9709, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 363718, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/examples/main.py', kind='file', lines=None, content=None, size_bytes=34, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 371723, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/scripts/bench_bm25_comparison.py', kind='file', lines=None, content=None, size_bytes=9406, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 475762,

In [4]:
preview(repo_glob("src/vfs/backends/*.py"), limit=25)


/repo/src/vfs/backends/__init__.py
/repo/src/vfs/backends/database.py
/repo/src/vfs/backends/mssql.py
/repo/src/vfs/backends/postgres.py


VFSResult(success=True, errors=[], function='glob', candidates=[Candidate(path='/repo/src/vfs/backends/__init__.py', kind='file', lines=None, content=None, size_bytes=260, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 588381, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/src/vfs/backends/database.py', kind='file', lines=None, content=None, size_bytes=120162, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 608611, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/src/vfs/backends/mssql.py', kind='file', lines=None, content=None, size_bytes=44882, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24, 1, 10, 30, 639714, tzinfo=datetime.timezone.utc)), Candidate(path='/repo/src/vfs/backends/postgres.py', kind='file', lines=None, content=None, size_bytes=54992, score=None, in_degree=None, out_degree=None, updated_at=datetime.datetime(2026, 4, 24,

In [5]:
# Example grep queries
preview(repo_grep("pg_trgm", max_count=25), limit=25)


/repo/context/stories/006-postgres-native-fts-lexical-search/spec.md:202:- **Typo / fuzzy matching (`pg_trgm`).**
/repo/context/stories/006-postgres-native-fts-lexical-search/spec.md:372:- Typo-tolerant retrieval via `pg_trgm`.
/repo/context/stories/007-postgres-glob-grep-regex-scaling/implementation.md:8:- native pattern search fail-fast requires `pg_trgm`, a partial `path text_pattern_ops` index, a partial `path gin_trgm_ops` index, and a partial `content gin_trgm_ops` index
/repo/context/stories/007-postgres-glob-grep-regex-scaling/implementation.md:41:- `_verify_pattern_schema(...)` rejects startup if `pg_trgm` is missing
/repo/context/stories/007-postgres-glob-grep-regex-scaling/implementation.md:115:- [`tests/conftest.py#L136-L200`](../../../tests/conftest.py#L136-L200) provisions the Postgres `search_tsv`, `pg_trgm`, `path text_pattern_ops`, `path gin_trgm_ops`, and `content gin_trgm_ops` artifacts for integration tests
/repo/context/stories/007-postgres-glob-grep-regex-scaling/

VFSResult(success=True, errors=[], function='grep', candidates=[Candidate(path='/repo/context/stories/006-postgres-native-fts-lexical-search/spec.md', kind='file', lines=[LineMatch(start=202, end=202, match=202), LineMatch(start=372, end=372, match=372)], content='# 006 — Postgres lexical search via native FTS (tsvector + GIN + ts_rank_cd)\n\n- **Status:** draft\n- **Date:** 2026-04-21\n- **Owner:** Clay Gendron\n- **Kind:** feature + backend\n\n## Intent\n\nReplace `PostgresFileSystem._lexical_search_impl`\'s mixed ranking path with a pure PostgreSQL native full-text search implementation:\n\n- A stored generated `search_tsv tsvector` column plus a partial `GIN` index owns recall.\n- A single SQL query ranks and limits results using `ts_rank_cd` with `[0, 1)`-bounded normalization.\n- The returned `Entry.score` is the `ts_rank_cd` value — no Python BM25 rerank step.\n\nThis is the right shape for standard PostgreSQL deployments: core PostgreSQL ships full-text search and `GIN`, but it

In [6]:
preview(
    repo_grep(
        "regex",
        globs=("**/*.py",),
        case_mode="smart",
        max_count=25,
    ),
    limit=25,
)


/repo/scripts/postgres_repo_cli_probe.py:552:        "grep --word-regexp does not treat beta_beta as bare beta",
/repo/scripts/postgres_repo_cli_probe.py:553:        f"grep {_quote('beta')} {notes_path} --word-regexp",
/repo/src/vfs/backends/database.py:76:def _regex_flags_for_mode(case_mode: CaseMode, pattern: str) -> int:
/repo/src/vfs/backends/database.py:89:def _compile_grep_regex(
/repo/src/vfs/backends/database.py:94:    word_regexp: bool,
/repo/src/vfs/backends/database.py:96:    """Build the effective grep regex from rg-style modifiers.
/repo/src/vfs/backends/database.py:104:    if word_regexp:
/repo/src/vfs/backends/database.py:106:    flags = _regex_flags_for_mode(case_mode, pattern)
/repo/src/vfs/backends/database.py:111:    """Extract guaranteed-literal alphanumeric runs from a regex.
/repo/src/vfs/backends/database.py:344:        downstream code (regex re-match, dedup, scoping) depends on it.
/repo/src/vfs/backends/database.py:366:        and helps regex assertions in ``te

VFSResult(success=True, errors=[], function='grep', candidates=[Candidate(path='/repo/scripts/postgres_repo_cli_probe.py', kind='file', lines=[LineMatch(start=552, end=553, match=552), LineMatch(start=552, end=553, match=553)], content='#!/usr/bin/env python3\n# ruff: noqa: E402, T201\n"""Probe an existing Postgres-backed repo snapshot through ``VFSClient.cli``.\n\nUsage:\n    uv run python scripts/postgres_repo_cli_probe.py\n    uv run python scripts/postgres_repo_cli_probe.py --strict-perf\n    ./scripts/postgres_repo_cli_probe.sh --keep-scratch\n\nThe script assumes the repository content is already loaded into a\nPostgreSQL database compatible with ``PostgresFileSystem``. It mounts that\ndatabase at ``/repo`` by default, then drives correctness, edge-case, and\nperformance probes through the public CLI query surface.\n\nWhen the corpus already contains native pgvector embeddings, the script also\nexecutes a direct vector-search smoke check through the public client API.\n"""\n\nfro

In [ ]:
# Edit these
GLOB_PATTERN = "**/*.md"
GREP_PATTERN = "authentication"

glob_result = repo_glob(GLOB_PATTERN, max_count=30)
grep_result = repo_grep(GREP_PATTERN, paths=(MOUNT,), max_count=30)

print("glob:")
preview(glob_result, limit=30)
print("\ngrep:")
preview(grep_result, limit=30)


In [ ]:
# Call this when you're done.
# client.close()
